In [2]:


import cv2
import os
import numpy as np
from PIL import Image
import json
import time

# Directories for datasets and trainer
dataset_path = "dataset"
trainer_path = "trainer"
names_file = "names.json"

# Create directories if they don't exist
if not os.path.exists(dataset_path):
    os.makedirs(dataset_path)
if not os.path.exists(trainer_path):
    os.makedirs(trainer_path)

# Load or initialize the names dictionary
if os.path.exists(names_file):
    with open(names_file, 'r') as file:
        names_dict = json.load(file)
else:
    names_dict = {}

# Step 1: Capture Training Data with Face Detection and Instructions
def capture_training_data(user_name):
    # Find user ID or create a new one
    user_id = None
    for key, value in names_dict.items():
        if value == user_name:
            user_id = key
            break

    if not user_id:
        user_id = str(len(names_dict) + 1)
        # Save the new name associated with this user ID
        names_dict[user_id] = user_name
        with open(names_file, 'w') as file:
            json.dump(names_dict, file)
        print(f"New user added: {user_name} with ID {user_id}")
    else:
        print(f"Adding more images for existing user: {user_name} with ID {user_id}")

    cap = cv2.VideoCapture(0)
    face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')
    sample_count = 0
    max_samples = 60  # Total number of new samples to capture

    # Instructions for user to look in different directions
    instructions = ["Look straight", "Turn left", "Turn right", "Tilt up", "Tilt down"]
    instruction_interval = max_samples // len(instructions)
    current_instruction = 0

    while True:
        ret, frame = cap.read()
        if not ret:
            print("Failed to capture frame from camera.")
            break

        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        faces = face_cascade.detectMultiScale(gray, 1.3, 5)

        # Show instructions on the screen for the user to follow
        if sample_count % instruction_interval == 0 and sample_count < max_samples:
            current_instruction = sample_count // instruction_interval
            instruction_text = instructions[current_instruction]
            # Display instruction for a brief moment before capturing
            cv2.putText(frame, instruction_text, (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 2)
            cv2.imshow("Capturing Training Data", frame)
            cv2.waitKey(2000)  # Wait for 2 seconds to allow the user to follow the instruction

        # Process each detected face
        for (x, y, w, h) in faces:
            sample_count += 1
            # Crop the face region
            face_image = gray[y:y+h, x:x+w]
            # Save cropped face image
            cv2.imwrite(f"{dataset_path}/User.{user_id}.{int(time.time())}_{sample_count}.jpg", face_image)
            cv2.rectangle(frame, (x, y), (x+w, y+h), (255, 0, 0), 2)

        # Display the frame with instructions and face bounding box
        cv2.imshow("Capturing Training Data", frame)

        # Stop after capturing max_samples samples
        if sample_count >= max_samples:
            print(f"Captured {sample_count} face images for {user_name}.")
            break

        if cv2.waitKey(1) & 0xFF == ord('q'):
            print("Capture process terminated by user.")
            break

    cap.release()
    cv2.destroyAllWindows()

# Step 2: Train the Model
def train_model():
    recognizer = cv2.face.LBPHFaceRecognizer_create()
    face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')

    # Function to get images and labels
    def get_images_and_labels(path):
        image_paths = [os.path.join(path, f) for f in os.listdir(path)]
        face_samples = []
        ids = []

        for image_path in image_paths:
            # Skip any non-image files
            if not image_path.lower().endswith(('.png', '.jpg', '.jpeg')):
                continue

            gray_image = Image.open(image_path).convert('L')
            image_np = np.array(gray_image, 'uint8')
            user_id = int(os.path.split(image_path)[-1].split('.')[1])
            faces = face_cascade.detectMultiScale(image_np)

            for (x, y, w, h) in faces:
                face_samples.append(image_np[y:y+h, x:x+w])
                ids.append(user_id)

        return face_samples, ids

    faces, ids = get_images_and_labels(dataset_path)
    recognizer.train(faces, np.array(ids))
    recognizer.write(f'{trainer_path}/trainer.yml')
    print("Model trained successfully!")

# Step 3: Recognize Faces
def recognize_faces():
    recognizer = cv2.face.LBPHFaceRecognizer_create()
    model_path = f'{trainer_path}/trainer.yml'

    # Check if the model file exists
    if not os.path.exists(model_path):
        print("Model not found. Please train the model first.")
        return

    recognizer.read(model_path)
    face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')

    cap = cv2.VideoCapture(0)
    confidence_threshold = 50  # Increased threshold for better accuracy

    while True:
        ret, frame = cap.read()
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        faces = face_cascade.detectMultiScale(gray, 1.3, 5)

        for (x, y, w, h) in faces:
            id, confidence = recognizer.predict(gray[y:y+h, x:x+w])

            # Calculate the confidence percentage
            confidence_percentage = 100 - confidence

            # Set a dynamic threshold based on the number of captured training samples
            if confidence_percentage > confidence_threshold:
                name = names_dict.get(str(id), "Unknown")
                text = f"{name}, {confidence_percentage:.2f}%"
                color = (0, 255, 0)  # Green for recognized faces
            else:
                text = "Unknown,  {:.2f}%".format(confidence_percentage)
                color = (0, 0, 255)  # Red for unknown faces

            cv2.putText(frame, text, (x, y-10), cv2.FONT_HERSHEY_SIMPLEX, 0.8, color, 2)
            cv2.rectangle(frame, (x, y), (x+w, y+h), color, 2)

        cv2.imshow("Face Recognition", frame)

        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    cap.release()
    cv2.destroyAllWindows()

# Main Program Execution
if __name__ == "__main__":
    choice = input("Enter 'capture' to capture training data, 'train' to train the model, or 'recognize' to recognize faces: ").strip().lower()
    if choice == 'capture':
        user_name = input("Enter User Name: ").strip()
        capture_training_data(user_name)
    elif choice == 'train':
        train_model()
    elif choice == 'recognize':
        recognize_faces()
    else:
        print("Invalid choice. Please enter 'capture', 'train', or 'recognize'.")




Enter 'capture' to capture training data, 'train' to train the model, or 'recognize' to recognize faces: recognize
